# Flash Flash Revolution Zero-Framer Processing - Silver Layer

This notebook resolves **"zero framers"** - consecutive notes with the same orientation that occur at identical timestamps. This is a data quality issue in the source where multiple notes should be temporally distinct but are recorded at the exact same moment, causing undefined reciprocal calculations in downstream density metrics.

## Problem Statement

When two notes of the same orientation (e.g., "0001") occur at the exact same timestamp:
- **Time delta** between them = 0
- **Vertical density** = 1/time_delta = **undefined (division by zero)**

This breaks downstream feature engineering for vertical and horizontal density calculations.

---

## Solution: Epsilon Adjustment

**Approach:**
1. Apply small epsilon value (1e-3 seconds) to separate sequential same-orientation notes
2. Use window function partitioned by (song_id, orientation)
3. If current note has same timestamp as previous note → add epsilon to current timestamp
4. This creates temporal ordering without affecting gameplay analysis

**Why This Works:**
- Epsilon (0.001 seconds) is below human perception threshold
- Maintains relative note order (note_id as tiebreaker)
- Creates valid time deltas for density calculations
- Preserves original timestamps for non-duplicate cases

---

## Note Aggregation

After epsilon adjustment, aggregate notes by (song_id, timestamp):
- Combines different orientations at the same timestamp into multi-orientation notes
- Example: "1000" + "0001" at timestamp=7.0 → "1001" at timestamp=7.0
- Uses element-wise OR to merge orientation bit strings
- Preserves minimal note_id and framers value per timestamp

**Why Aggregate:**
- Reflects actual gameplay: multi-arrow notes hit simultaneously
- Reduces redundancy from epsilon adjustment applied to different orientations
- Output table represents true note timing structure

---

## Output Schema

**Table:** `acubed.ffr.silver__notes-adjusted`

**Columns:**
- `song_id` (bigint): Unique song identifier
- `note_id` (int): Minimal note_id for notes at this timestamp
- `framers` (int): Minimal framers value for notes at this timestamp
- `orientation` (string): Aggregated multi-orientation (e.g., "1001", "1110")
- `timestamp` (double): Epsilon-adjusted timestamp

**Data Quality:**
- All orientation strings contain only 0 and 1 (validated)
- No zero-framer issues remain
- All notes temporally ordered within each orientation partition

---

## Source Tables

**Silver:**
- `acubed.ffr.silver__notes` - Note-level data with orientation and timestamp

In [0]:
# Epsilon value to add to timestamps with zero delta
# Adjust this value to control the timestamp increment
epsilon = 1e-3  # 0.001 seconds

In [0]:
from pyspark.sql import Window
from pyspark.sql.functions import lag, col, when

# Read source table
df = spark.table("acubed.ffr.silver__notes")

# Define window spec: partition by song_id and orientation, order by timestamp and note_id
window_spec = Window.partitionBy("song_id", "orientation").orderBy("timestamp", "note_id")

# Add previous timestamp and adjust to separate sequential notes of same orientation
df_epsilon_adjusted = (
    df
    .withColumn("prev_timestamp", lag("timestamp").over(window_spec))
    .withColumn(
        "timestamp",
        when(
            (col("prev_timestamp").isNotNull()) & 
            (col("timestamp") - col("prev_timestamp") == 0),
            col("timestamp") + epsilon
        ).otherwise(col("timestamp"))
    )
    .drop("prev_timestamp")
)

print(f"✓ Applied epsilon adjustment to separate sequential notes")
print(f"Original row count: {df.count()}")
print(f"Epsilon-adjusted row count: {df_epsilon_adjusted.count()}")

print("\nSample of epsilon-adjusted notes:")
display(df_epsilon_adjusted.orderBy("song_id", "timestamp", "note_id").limit(10))

In [0]:
from pyspark.sql.functions import col, collect_list, min as min_, udf
from pyspark.sql.types import StringType

# UDF to combine orientations (element-wise OR of binary strings)
@udf(returnType=StringType())
def combine_orientations(orientation_list):
    """Combine multiple orientation strings by OR-ing each position.
    Example: ['1000', '0001'] -> '1001'
             ['1100', '0010'] -> '1110'
    """
    if not orientation_list:
        return '0000'
    
    # Convert to list of lists of integers
    orientation_arrays = [[int(c) for c in orient] for orient in orientation_list]
    
    # Element-wise OR across all orientations
    result = [0, 0, 0, 0]
    for orient_array in orientation_arrays:
        for i in range(4):
            result[i] = result[i] or orient_array[i]
    
    return ''.join(str(x) for x in result)

# Aggregate notes with same song_id and timestamp (after epsilon adjustment)
df_aggregated = (
    df_epsilon_adjusted
    .groupBy("song_id", "timestamp")
    .agg(
        min_("note_id").alias("note_id"),  # Keep the minimum note_id
        min_("framers").alias("framers"),  # Keep the minimum framers
        combine_orientations(collect_list("orientation")).alias("orientation")
        # color column is dropped (not included in agg)
    )
    .select("song_id", "note_id", "framers", "orientation", "timestamp")
    .orderBy("song_id", "timestamp", "note_id")
)

# Write to adjusted table
df_aggregated.write.mode("overwrite").saveAsTable("acubed.ffr.`silver__notes-adjusted`")

print(f"✓ Created table acubed.ffr.silver__notes-adjusted")
print(f"Epsilon-adjusted row count: {df_epsilon_adjusted.count()}")
print(f"Final aggregated row count: {df_aggregated.count()}")

# Check for any '2', '3', etc. in orientation strings
invalid_orientations = df_aggregated.filter(
    col("orientation").rlike("[2-9]")
).count()

print(f"\nVerification - Notes with invalid orientation digits (2-9): {invalid_orientations}")

print("\nSample of final aggregated notes:")
display(df_aggregated.limit(10))

In [0]:
from pyspark.sql.functions import count, col

# Find original cases where same orientation appeared multiple times at same timestamp
original_same_orientation = (
    spark.table("acubed.ffr.silver__notes")
    .groupBy("song_id", "timestamp", "orientation")
    .agg(count("*").alias("note_count"))
    .filter(col("note_count") > 1)
    .orderBy(col("note_count").desc())
    .limit(5)
)

# print("Examples where same orientation appeared multiple times at same timestamp:")
# display(original_same_orientation)

if original_same_orientation.count() > 0:
    sample_row = original_same_orientation.first()
    sample_song_id = sample_row['song_id']
    sample_timestamp = sample_row['timestamp']
    sample_orientation = sample_row['orientation']
    
    print(f"\nDetailed example: song_id={sample_song_id}, timestamp={sample_timestamp}, orientation={sample_orientation}")
    
    print("\n1. Original notes (BEFORE epsilon adjustment):")
    display(
        spark.table("acubed.ffr.silver__notes")
        .filter(
            (col("song_id") == sample_song_id) & 
            (col("timestamp") == sample_timestamp)
        )
        .select("song_id", "note_id", "orientation", "timestamp")
        .orderBy("note_id")
    )
    
    print("\n2. After epsilon adjustment (timestamps are now different):")
    # Need to check a range since epsilon was applied
    display(
        df_epsilon_adjusted
        .filter(
            (col("song_id") == sample_song_id) & 
            (col("timestamp") >= sample_timestamp) &
            (col("timestamp") < sample_timestamp + 0.01)
        )
        .select("song_id", "note_id", "orientation", "timestamp")
        .orderBy("timestamp", "note_id")
    )
    
    print("\n3. Final result after aggregation (each note preserved at different timestamp):")
    display(
        spark.table("acubed.ffr.`silver__notes-adjusted`")
        .filter(
            (col("song_id") == sample_song_id) & 
            (col("timestamp") >= sample_timestamp) &
            (col("timestamp") < sample_timestamp + 0.01)
        )
        .select("song_id", "note_id", "orientation", "timestamp")
        .orderBy("timestamp", "note_id")
    )
    
    print(f"\n✓ Without epsilon-first approach: would have combined to orientation with '2'")
    print(f"✓ With epsilon-first approach: preserved as separate notes at different timestamps")